In [ ]:
# Colab (Python 3.12) setup:
# 1) Mount Google Drive
# 2) Clone RDT repo
# 3) Install deps (minimal)
# 4) Download/load T5-XXL via the repo's T5Embedder
# 5) Compute + save embeddings for your 5 instructions
# 6) Store embeddings folder on Google Drive and show how to load it

import os
import shutil
from pathlib import Path

In [ ]:
# ----------------------------
# 0) Google Drive mount
# ----------------------------
from google.colab import drive
drive.mount("/content/drive")

# Change this if you want a different location on Drive
DRIVE_OUT_DIR = Path("/content/drive/MyDrive/rdt_lang_embeddings_maniskill")
DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional: if you use gated HF models, set token in Colab Secrets or here:
# os.environ["HUGGINGFACE_HUB_TOKEN"] = "hf_..."

Mounted at /content/drive


In [ ]:
# ----------------------------
# 1) Clone repo
# ----------------------------
REPO_DIR = Path("/content/RoboticsDiffusionTransformer")
if not REPO_DIR.exists():
    !git clone https://github.com/thu-ml/RoboticsDiffusionTransformer.git /content/RoboticsDiffusionTransformer

os.chdir(str(REPO_DIR))

Cloning into '/content/RoboticsDiffusionTransformer'...
remote: Enumerating objects: 350, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 350 (delta 115), reused 86 (delta 83), pack-reused 186 (from 2)
Receiving objects: 100% (350/350), 4.55 MiB | 19.91 MiB/s, done.
Resolving deltas: 100% (181/181), done.


In [ ]:
# ----------------------------
# 2) Install dependencies
# ----------------------------
# Notes:
# - Colab already has torch; we try to keep changes minimal.
# - If you need a specific torch version, pin it here (may take time and can break other libs).
!pip -q install --upgrade pip
!pip -q install packaging==24.0 pyyaml huggingface_hub safetensors sentencepiece
!pip -q install -r requirements.txt

# flash-attn often fails on Colab; only try if you know it works in your runtime:
# !pip -q install flash-attn --no-build-isolation

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 83.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-bigquery 3.40.0 requires packaging>=24.2.0, but you have packaging 24.0 which is incompatible.
xarray 2025.12.0 requires packaging>=24.1, but you have packaging 24.0 which is incompatible.
db-dtypes 1.5.0 requires packaging>=24.2.0, but you have packaging 24.0 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.8 which is incompatible.
grain 0.2.15 requires protobuf>=5

In [ ]:
# ----------------------------
# 3) Compute embeddings
# ----------------------------
import torch
import yaml

from models.multimodal_encoder.t5_encoder import T5Embedder

GPU = 0
MODEL_PATH = "google/t5-v1_1-xxl"
CONFIG_PATH = "configs/base.yaml"

# Local scratch output; we'll copy to Drive at the end
LOCAL_SAVE_DIR = Path("/content/rdt_lang_embeddings_tmp")
if LOCAL_SAVE_DIR.exists():
    shutil.rmtree(LOCAL_SAVE_DIR)
LOCAL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Offload directory (recommended if VRAM < 24GB)
OFFLOAD_DIR = Path("/content/rdt_t5_offload")
OFFLOAD_DIR.mkdir(parents=True, exist_ok=True)

TASKS = [
    ("PushCube-v1", "Push the blue cube towards the target"),
    ("PickCube-v1", "Pick up the red cube and bring it to the green goal"),
    ("StackCube-v1", "Pick up the red cube and stack it on top of the green cube"),
    ("PegInsertionSide-v1", "Pick up the peg and insert it into the hole"),
    ("PullCubeTool-v1", "Use the hook tool to pull the blue cube towards the robot arm base"),
]

with open(CONFIG_PATH, "r") as fp:
    config = yaml.safe_load(fp)

device = torch.device(f"cuda:{GPU}" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Create embedder (this will download weights from HF if not present)
text_embedder = T5Embedder(
    from_pretrained=MODEL_PATH,
    model_max_length=config["dataset"]["tokenizer_max_length"],
    device=device,
    use_offload_folder=str(OFFLOAD_DIR) if OFFLOAD_DIR is not None else None,
)
tokenizer, text_encoder = text_embedder.tokenizer, text_embedder.model
text_encoder.eval()

def encode_one(name: str, instruction: str) -> Path:
    tokens = tokenizer(
        instruction,
        return_tensors="pt",
        padding="longest",
        truncation=True,
    )["input_ids"].to(device)

    tokens = tokens.view(1, -1)
    with torch.no_grad():
        emb = text_encoder(tokens).last_hidden_state.detach().cpu()

    save_path = LOCAL_SAVE_DIR / f"{name}.pt"
    torch.save(
        {
            "name": name,
            "instruction": instruction,
            "embeddings": emb,
        },
        save_path,
    )
    print(f"[OK] {name}: {emb.shape} -> {save_path}")
    return save_path

saved = []
for name, instr in TASKS:
    saved.append(encode_one(name, instr))

# Optional: save an index file for convenience
index_path = LOCAL_SAVE_DIR / "index.txt"
with open(index_path, "w", encoding="utf-8") as f:
    for name, instr in TASKS:
        f.write(f"{name}\t{instr}\n")
print(f"[OK] Wrote index: {index_path}")

Device: cuda:0


tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/593 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


pytorch_model.bin:   0%|          | 0.00/44.5G [00:00<?, ?B/s]


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.12/dist-package

AttributeError: _ARRAY_API not found

[OK] PushCube-v1: torch.Size([1, 9, 4096]) -> /content/rdt_lang_embeddings_tmp/PushCube-v1.pt
[OK] PickCube-v1: torch.Size([1, 14, 4096]) -> /content/rdt_lang_embeddings_tmp/PickCube-v1.pt
[OK] StackCube-v1: torch.Size([1, 17, 4096]) -> /content/rdt_lang_embeddings_tmp/StackCube-v1.pt
[OK] PegInsertionSide-v1: torch.Size([1, 12, 4096]) -> /content/rdt_lang_embeddings_tmp/PegInsertionSide-v1.pt
[OK] PullCubeTool-v1: torch.Size([1, 16, 4096]) -> /content/rdt_lang_embeddings_tmp/PullCubeTool-v1.pt
[OK] Wrote index: /content/rdt_lang_embeddings_tmp/index.txt


In [ ]:
!ls -lh /content/rdt_lang_embeddings_tmp


total 568K
-rw-r--r-- 1 root root  333 Feb  8 16:39 index.txt
-rw-r--r-- 1 root root  98K Feb  8 16:39 PegInsertionSide-v1.pt
-rw-r--r-- 1 root root 114K Feb  8 16:39 PickCube-v1.pt
-rw-r--r-- 1 root root 130K Feb  8 16:39 PullCubeTool-v1.pt
-rw-r--r-- 1 root root  74K Feb  8 16:39 PushCube-v1.pt
-rw-r--r-- 1 root root 138K Feb  8 16:39 StackCube-v1.pt


In [ ]:
!cat /content/rdt_lang_embeddings_tmp/index.txt


PushCube-v1	Push the blue cube towards the target
PickCube-v1	Pick up the red cube and bring it to the green goal
StackCube-v1	Pick up the red cube and stack it on top of the green cube
PegInsertionSide-v1	Pick up the peg and insert it into the hole
PullCubeTool-v1	Use the hook tool to pull the blue cube towards the robot arm base


In [ ]:
# ----------------------------
# 4) Copy embeddings folder to Google Drive
# ----------------------------
# Copy the whole folder to Drive (overwrite existing)
if DRIVE_OUT_DIR.exists():
    shutil.rmtree(DRIVE_OUT_DIR)
shutil.copytree(LOCAL_SAVE_DIR, DRIVE_OUT_DIR)
print(f"[OK] Copied embeddings to Drive: {DRIVE_OUT_DIR}")

[OK] Copied embeddings to Drive: /content/drive/MyDrive/rdt_lang_embeddings_maniskill


In [ ]:
# ----------------------------
# 5) Example: load embeddings folder from Drive
# ----------------------------
def load_all_embeddings(folder: Path):
    out = {}
    for pt in folder.glob("*.pt"):
        d = torch.load(pt, map_location="cpu")
        out[d["name"]] = d
    return out

loaded = load_all_embeddings(DRIVE_OUT_DIR)
print("[OK] Loaded keys:", list(loaded.keys()))
print("[Example] One entry shapes:", {k: v["embeddings"].shape for k, v in list(loaded.items())[:1]})

[OK] Loaded keys: ['PushCube-v1', 'StackCube-v1', 'PegInsertionSide-v1', 'PickCube-v1', 'PullCubeTool-v1']
[Example] One entry shapes: {'PushCube-v1': torch.Size([1, 9, 4096])}
